# Data preparation

Data splitting, feature scaling and saving the datasets for modeling.

## Steps

1. Load and validate the data
2. Split into train and test
3. Scale the numeric features
4. Validate and save the datasets

In [1]:
# Load and validate the data
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/processed/produkcja_clean.csv')

display(df.shape)
display(df.dtypes)
display(df.isna().sum())
display(df.duplicated().sum())
display(df['Machine failure'].value_counts())

(10000, 9)

Type_L                       int64
Type_M                       int64
Process temperature [K]    float64
Temperature difference     float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Power [W]                  float64
Tool wear [min]              int64
Machine failure              int64
dtype: object

Type_L                     0
Type_M                     0
Process temperature [K]    0
Temperature difference     0
Rotational speed [rpm]     0
Torque [Nm]                0
Power [W]                  0
Tool wear [min]            0
Machine failure            0
dtype: int64

np.int64(0)

Machine failure
0    9670
1     330
Name: count, dtype: int64

In [2]:
# Separate features and target
X = df.drop(columns = ['Machine failure'])
y = df['Machine failure']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("Class distribution:")
print(y.value_counts())
print(y.value_counts(normalize=True).round(3) * 100)

X shape: (10000, 8)
y shape: (10000,)
Class distribution:
Machine failure
0    9670
1     330
Name: count, dtype: int64
Machine failure
0    96.7
1     3.3
Name: proportion, dtype: float64


In [3]:
# Split into train and test
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
y_train = y_train_raw.copy()
y_test = y_test_raw.copy()

assert X_train_raw.index.equals(y_train_raw.index), "X_train and y_train indexes are not aligned"
assert X_test_raw.index.equals(y_test_raw.index), "X_test and y_test indexes are not aligned"
assert len(X_train_raw) == len(y_train_raw), "X_train_raw and y_train_raw have different row counts."
assert len(X_test_raw) == len(y_test_raw), "X_test_raw and y_test_raw have different row counts."
assert np.isclose(y_train_raw.mean(), y_test_raw.mean(), atol = 0.001), "Class proportions in train and test differ more than expected."
print(f"Train : {X_train.shape[0]} records")
print(f"Test : {X_test.shape[0]} records")
print(f"Failures in train: {y_train.sum()} ({y_train.mean() * 100:.1f}%)")
print(f"Failures in test: {y_test.sum()} ({y_test.mean() * 100:.1f}%)")
print("Split validation completed successfully.")
print(f"Number of training records: {len(X_train_raw)}")
print(f"Number of test records: {len(X_test_raw)}")

Train : 8000 records
Test : 2000 records
Failures in train: 264 (3.3%)
Failures in test: 66 (3.3%)
Split validation completed successfully.
Number of training records: 8000
Number of test records: 2000


In [4]:
# Scale the numeric features
cols_to_scale = [
    'Process temperature [K]',
    'Temperature difference',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Power [W]',
    'Tool wear [min]'
]

scaler = StandardScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train_raw[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test_raw[cols_to_scale])

assert X_train.index.equals(X_train_raw.index), "Scaling changed the training-set index."

assert X_test.index.equals(X_test_raw.index), "Scaling changed the test-set index."

assert X_train.shape == X_train_raw.shape, ("Scaled and raw training sets have different shapes.")

assert X_test.shape == X_test_raw.shape, ("Scaled and raw test sets have different shapes.")

assert (y_test.reset_index(drop=True) == y_test_raw.reset_index(drop=True)).all(), "Target values in the two test sets differ."

assert (y_train.reset_index(drop=True) == y_train_raw.reset_index(drop=True)).all(), "Target values in the two training sets differ."

print("Scaling validation completed successfully.")

Scaling validation completed successfully.


In [5]:
# Scaling summary
scaled_df = pd.DataFrame(
    X_train[cols_to_scale],
    columns = cols_to_scale
)
display(scaled_df.describe().round(3))

,Process temperature [K],Temperature difference,Rotational speed [rpm],Torque [Nm],Power [W],Tool wear [min]
count,8000.000,8000.000,8000.000,8000.000,8000.000,8000.000
mean,-0.000,0.000,-0.000,-0.000,-0.000,0.000
std,1.000,1.000,1.000,1.000,1.000,1.000
min,-2.908,-2.401,-2.072,-3.631,-4.803,-1.694
25%,-0.814,-0.700,-0.648,-0.681,-0.672,-0.859
50%,0.064,-0.200,-0.201,0.011,-0.005,-0.009
75%,0.739,1.000,0.404,0.684,0.676,0.856
max,2.563,2.100,7.526,3.633,3.699,2.289


In [6]:
# Scaler parameters
print(scaler.mean_)
print(scaler.scale_)

[ 310.005375    10.0002375 1538.92525     39.98705   6280.70232
  107.59825  ]
[1.48058303e+00 9.99674294e-01 1.78987704e+02 9.96722403e+00
 1.06853168e+03 6.35351269e+01]


In [7]:
# Validate and save the datasets

assert X_train_raw.index.equals(
    y_train.index
), "X_train_raw and y_train indexes are not aligned."

assert X_test_raw.index.equals(
    y_test.index
), "X_test_raw and y_test indexes are not aligned."

assert X_train.index.equals(
    X_train_raw.index
), "X_train and X_train_raw have different indexes."

assert X_test.index.equals(
    X_test_raw.index
), "X_test and X_test_raw have different indexes."


# Unscaled data
X_train_raw.to_csv(
    "../data/processed/X_train_raw.csv",
    index=True,
    index_label="record_index"
)

X_test_raw.to_csv(
    "../data/processed/X_test_raw.csv",
    index=True,
    index_label="record_index"
)


# Scaled data
X_train.to_csv(
    "../data/processed/X_train_scaled.csv",
    index=True,
    index_label="record_index"
)

X_test.to_csv(
    "../data/processed/X_test_scaled.csv",
    index=True,
    index_label="record_index"
)


# Target
y_train.to_csv(
    "../data/processed/y_train.csv",
    index=True,
    index_label="record_index"
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=True,
    index_label="record_index"
)

print("All datasets were saved successfully.")

All datasets were saved successfully.


## Observations

- The data is complete and free of duplicates.
- The split preserves the class proportions.
- Scaling was fit on the training set only.
- `record_index` keeps the link to the original record.